<a href="https://colab.research.google.com/github/tiamrit604-pixel/PDM_2026/blob/main/kaggle_api_download_data_updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Collection: Kaggle API & Direct Downloads
---
## Learning Objectives
By the end of this notebook you will be able to:
1. Install and configure the Kaggle API in Google Colab
2. Securely set up your Kaggle credentials
3. Search, browse, and download datasets from Kaggle
4. Load downloaded data into a pandas DataFrame
5. Download data from direct URLs as an alternative method




---
## What is the Kaggle API?
Kaggle is the world's largest data science community, hosting **50,000+ public datasets**.  
The **Kaggle API** lets you download datasets directly from a notebook — no manual clicking needed.

```
Your Colab Notebook  ──(API request)──▶  Kaggle Servers  ──▶  Dataset files
```

**Why use it?**
- Reproducible: anyone can re-run your notebook and get the same data
- Fast: download large datasets in one command
- Automated: works inside pipelines and scheduled notebooks


---
##  Step 1: Install the Kaggle Library
The `!` prefix tells Colab to run a **shell command** instead of Python.  
`--quiet` suppresses the long install log so output stays clean.


In [ ]:
# Install the Kaggle Python package
# --quiet hides verbose output; remove it if you want to see what's being installed
!pip install kaggle --quiet
print("Kaggle installed successfully!")

Kaggle installed successfully!


---
## Step 2: Create Your Kaggle Account & Download Your API Token

Follow these steps **before running the next cell**:

| # | Action | Where |
|---|--------|-------|
| 1 | Create a free account | [kaggle.com/account/login](https://www.kaggle.com/account/login) |
| 2 | Go to your Account settings | [kaggle.com/settings](https://www.kaggle.com/settings) |
| 3 | Scroll to **"API"** section | Under *Legacy API Credentials* |
| 4 | Click **"Create New Legacy API Key"** | This downloads a `kaggle.json` file |
| 5 | Keep that file — you'll upload it next | |

> 🔒 **Security note:** Your `kaggle.json` contains your personal API key.  
> Never share it, commit it to GitHub, or paste it in a public notebook!


---
##  Step 3: Upload Your `kaggle.json` to Colab
Run the cell below — a file picker will appear. Select your downloaded `kaggle.json`.


In [ ]:
# This uploads kaggle.json from your local computer into the Colab session
from google.colab import files

print(" Please select your kaggle.json file...")
uploaded = files.upload()

# Confirm what was uploaded
for filename in uploaded.keys():
    print(f"Uploaded: {filename} ({len(uploaded[filename])} bytes)")

 Please select your kaggle.json file...


Saving kaggle.json to kaggle.json
Uploaded: kaggle.json (65 bytes)


---
## Step 4: Move Credentials to the Right Location
The Kaggle CLI expects your credentials in `~/.kaggle/kaggle.json`.  
We'll also set the correct file permissions so only your user can read it.


In [ ]:
import os

# Create the hidden .kaggle directory in the home folder
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)

# Move the uploaded file to the correct location
!mv kaggle.json ~/.kaggle/kaggle.json

# Set permissions: owner read/write only (required by Kaggle CLI)
!chmod 600 ~/.kaggle/kaggle.json

print("kaggle.json moved to ~/.kaggle/")
print("File permissions set to 600 (owner read/write only)")

kaggle.json moved to ~/.kaggle/
File permissions set to 600 (owner read/write only)


---
##  Step 5: Verify Your Setup
Let's confirm the Kaggle CLI is working and authenticated.


In [ ]:
# Check installed version
!kaggle --version

# List your Kaggle username (confirms credentials are working)
print("\n--- Testing authentication ---")
!kaggle config view

Kaggle CLI 2.0.2

--- Testing authentication ---
Configuration values from /root/.kaggle
- username: toystory2
- auth_method: LEGACY_API_KEY
- path: None
- proxy: None
- competition: None


---
##  Step 6: Explore Available Datasets
Before downloading, let's browse what's available.  

`!kaggle datasets list` returns the most popular datasets.  
You can filter with flags like `--search "housing"` or `--sort-by votes`.


In [ ]:
# List the top 10 most popular datasets on Kaggle
print(" Top Kaggle Datasets:")
!kaggle datasets list --sort-by votes --max-size 50

 Top Kaggle Datasets:
ref                                                               title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
----------------------------------------------------------------  --------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
asaniczka/tmdb-movies-dataset-2023-930k-movies                    Full TMDB Movies Dataset 2024 (1M Movies)            255789082  2026-05-28 06:58:54.310000          43212        808  0.88235295       
rsrishav/youtube-trending-video-dataset                           YouTube Trending Video Dataset (updated daily)               0  2024-04-15 12:27:48.547000          42045        627  1                
teocalvo/teomewhy-loyalty-system                                  TeoMeWhy Loyalty System                               59131452  2026-05-28 12:32:57.600000          1361

In [ ]:
# Search for a specific topic
print("Searching for 'housing price' datasets:")
!kaggle datasets list --search "housing price" --max-size 20

Searching for 'housing price' datasets:
ref                                     title                                                   size  lastUpdated                 downloadCount  voteCount  usabilityRating  
--------------------------------------  ------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
lorentzyeung/price-paid-data-202304     UK Property Price official data (Monthly Update)   956418113  2026-04-30 13:53:39.033000           2653         28  0.7058824        
sethdowden/portland-maps-assessor-data  Portland Assessor Data                              47620259  2026-05-26 04:14:33.487000           1065         21  1                


---
##  Step 7: Download a Dataset
We'll download the **Iris flower dataset** — a classic data science starter dataset.  
It contains 150 rows with measurements for 3 species of iris flowers.

**Command format:**  
`!kaggle datasets download -d <owner>/<dataset-name>`

You can find the owner/name from the dataset's Kaggle URL.


In [ ]:
import os

# Create a folder to store our downloaded data
os.makedirs("data", exist_ok=True)

print(" Downloading the Iris dataset from Kaggle...")
!kaggle datasets download -d uciml/iris -p data/ --unzip --quiet

# Confirm the files were downloaded
print("\n Files in data/ folder:")
for f in os.listdir("data"):
    size = os.path.getsize(f"data/{f}")
    print(f"  {f}  ({size:,} bytes)")

Dataset URL: https://www.kaggle.com/datasets/uciml/iris
License(s): CC0-1.0

 Files in data/ folder:
  Iris.csv  (5,107 bytes)
  database.sqlite  (10,240 bytes)


---
## Step 8: Load & Preview the Data
Now let's actually use the data we downloaded!


In [ ]:
import pandas as pd

# Load the downloaded CSV into a DataFrame
df = pd.read_csv("data/Iris.csv")

print(f" Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print("\n--- First 5 rows ---")
df.head()

 Dataset loaded: 150 rows × 6 columns

--- First 5 rows ---


,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


In [ ]:
# Quick summary statistics
print("--- Dataset Info ---")
print(df.dtypes)
print("\n--- Basic Statistics ---")
df.describe()

--- Dataset Info ---
Id                 int64
SepalLengthCm    float64
SepalWidthCm     float64
PetalLengthCm    float64
PetalWidthCm     float64
Species           object
dtype: object

--- Basic Statistics ---


,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm
count,150.000000,150.000000,150.000000,150.000000,150.000000
mean,75.500000,5.843333,3.054000,3.758667,1.198667
std,43.445368,0.828066,0.433594,1.764420,0.763161
min,1.000000,4.300000,2.000000,1.000000,0.100000
25%,38.250000,5.100000,2.800000,1.600000,0.300000
50%,75.500000,5.800000,3.000000,4.350000,1.300000
75%,112.750000,6.400000,3.300000,5.100000,1.800000
max,150.000000,7.900000,4.400000,6.900000,2.500000


In [ ]:
# How many flowers of each species?
print("🌸 Species distribution:")
print(df['Species'].value_counts())

# Simple group-by example
print("\n Average petal length by species:")
print(df.groupby('Species')['PetalLengthCm'].mean().round(2))

🌸 Species distribution:
Species
Iris-setosa        50
Iris-versicolor    50
Iris-virginica     50
Name: count, dtype: int64

 Average petal length by species:
Species
Iris-setosa        1.46
Iris-versicolor    4.26
Iris-virginica     5.55
Name: PetalLengthCm, dtype: float64


---
## Bonus Method: Download Data via Direct URL
Sometimes data isn't on Kaggle. Many organizations publish open data via direct download links.  
`wget` is a command-line tool for downloading files from URLs.

We'll download **Austin Airbnb listings** — real rental data from [insideairbnb.com](https://insideairbnb.com).

> 💡 Visit [insideairbnb.com/get-the-data](https://insideairbnb.com/get-the-data/) to find the latest dataset URL for any city.


In [ ]:
import pandas as pd

# Direct URL for Austin Airbnb summary listings data
url = "https://data.insideairbnb.com/united-states/tx/austin/2025-09-16/visualisations/listings.csv"

# Load the dataset directly into pandas
airbnb_df = pd.read_csv(url)

# Preview the first few rows
airbnb_df.head()


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license
0,5456,"Walk to 6th, Rainey St and Convention Ctr",8028,Sylvia,NaN,78702,30.26057,-97.73441,Entire home/apt,97.0,2,708,2025-09-02,3.52,1,328,25,NaN
1,6448,"Secluded Studio @ Zilker - King Bed, Bright & ...",14156,Amy,NaN,78704,30.26034,-97.76487,Entire home/apt,160.0,3,339,2025-08-20,1.98,1,316,14,NaN
2,8502,Woodland Studio Lodging,25298,Karen,NaN,78741,30.23466,-97.73682,Entire home/apt,38.0,4,54,2025-05-05,0.28,1,88,1,NaN
3,13035,Historic house in highly walkable East Austin,50793,Margaret Ann,NaN,78702,30.26098,-97.73072,Entire home/apt,145.0,15,19,2025-07-31,0.11,2,321,1,NaN
4,22828,Garage Apartment central SE Austin,56488,David,NaN,78741,30.23614,-97.73225,Entire home/apt,58.0,30,56,2025-08-16,0.30,1,211,3,NaN


---
## **Practice Exercise**

2 points

Try these on your own:

1. **Search** for a dataset on Kaggle related to your interests (`!kaggle datasets list --search "your topic"`)
2. **Download** it using `!kaggle datasets download -d owner/dataset-name -p data/ --unzip`
3. **Load** it with `pd.read_csv()` and print `.shape`

---
## Summary

| What you did | Command used |
|---|---|
| Installed Kaggle | `!pip install kaggle` |
| Set up credentials | Uploaded `kaggle.json`, moved to `~/.kaggle/` |
| Browsed datasets | `!kaggle datasets list --search "topic"` |
| Downloaded dataset | `!kaggle datasets download -d owner/name` |
| Downloaded via URL | `!wget <url>` + `!gzip -d file.csv.gz` |
| Loaded into Python | `pd.read_csv("file.csv")` |

> 🔁 **Remember:** Colab sessions reset! You'll need to re-upload `kaggle.json` each time you start a new session.  
> **Tip:** Store it in Google Drive and copy it at session start to avoid re-uploading every time.


Download All Austin Airbnb CSV Files:

3 points


In this exercise, you will download all available CSV files for Austin Airbnb data from Inside Airbnb.

Inside Airbnb currently lists Austin files such as:

- listings.csv.gz — detailed listings data
- calendar.csv.gz — detailed calendar data
- reviews.csv.gz — detailed reviews data
- listings.csv — summary listings data, useful for visualization

*The .csv.gz files are compressed CSV files, and pandas can read them directly. Inside Airbnb lists these Austin files on its Get the Data page.*



In [ ]:
#starter code for you

import os
import requests

# Step 1: Create a folder to store the downloaded files
folder_name = "austin_airbnb_data"
os.makedirs(folder_name, exist_ok=True)